# Legend Pattern Search on a PDF Page

Standalone prototype. Finds every region on a single PDF page that is filled
with the same hatching texture as a named legend swatch and exports:

- an annotated PNG of the page with translucent-green highlights + polygon outlines
- a JSON of polygon vertices in PDF point space

Algorithm: render page at high DPI, crop swatch from the same render, binarize
both, run `cv2.matchTemplate` with `TM_CCOEFF_NORMED`, paint template footprints
at every hit, close gaps, then extract contours.

Knobs at the top of Cell 1 (`DPI`, `BIN_THRESHOLD`, `MATCH_THRESHOLD`, etc.) are
the only things you should need to tune.

In [ ]:
import json
import re
from pathlib import Path

import cv2
import fitz
import numpy as np
from PIL import Image

# ---------- Inputs ----------
drawing = "d3"
page_num = 26
template_name = 'C1 VARIES 5/8" GYPSUM BD ON 3-5/8"'

# ---------- Knobs ----------
DPI = 300
BIN_THRESHOLD = 200
MATCH_THRESHOLD = 0.60
MIN_POLY_AREA_FRAC = 0.5
APPROX_EPS_PX = 1.5
CLOSE_KERNEL = 3
CLOSE_ITERS = 2

# ---------- Paths ----------
pdf_path = f"../data/{drawing}/{drawing}.pdf"
legends_path = f"../data/{drawing}/legends/new_v2/{drawing}_legends_{page_num}.json"
out_dir = Path(f"../data/{drawing}/legends/matches")
out_dir.mkdir(parents=True, exist_ok=True)

# ---------- Load swatch rect ----------
# Supports both a flat dict {name: {x0,top,x1,bottom}} and a list of single-key
# dicts (which is what the older legends/ folder used).
raw = json.load(open(legends_path))
if isinstance(raw, list):
    flat = {}
    for entry in raw:
        flat.update(entry)
    rect_data = flat[template_name]
else:
    rect_data = raw[template_name]

# ---------- Render page ----------
scale = DPI / 72.0
doc = fitz.open(pdf_path)
page = doc[page_num]
pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
page_rgb = (
    np.frombuffer(pix.samples, dtype=np.uint8)
    .reshape(pix.height, pix.width, 3)
    .copy()
)

# ---------- Crop template from the rendered page ----------
tx0 = int(round(rect_data["x0"] * scale))
ty0 = int(round(rect_data["top"] * scale))
tx1 = int(round(rect_data["x1"] * scale))
ty1 = int(round(rect_data["bottom"] * scale))
template_rgb = page_rgb[ty0:ty1, tx0:tx1].copy()
th, tw = template_rgb.shape[:2]

slug = re.sub(r"[^A-Za-z0-9]+", "_", template_name).strip("_").lower()
Image.fromarray(template_rgb).save(out_dir / f"{drawing}_p{page_num}_{slug}_template.png")

print(f"Page render : {page_rgb.shape[1]}x{page_rgb.shape[0]} px  (scale={scale:.3f})")
print(f"Template box: ({tx0},{ty0})-({tx1},{ty1})  size={tw}x{th} px")
print(f"Saved debug template to {out_dir / f'{drawing}_p{page_num}_{slug}_template.png'}")

In [ ]:
page_gray = cv2.cvtColor(page_rgb, cv2.COLOR_RGB2GRAY)
template_gray = cv2.cvtColor(template_rgb, cv2.COLOR_RGB2GRAY)

# Ink = 255, paper = 0 after THRESH_BINARY_INV
_, page_bin = cv2.threshold(page_gray, BIN_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
_, tmpl_bin = cv2.threshold(template_gray, BIN_THRESHOLD, 255, cv2.THRESH_BINARY_INV)

result = cv2.matchTemplate(page_bin, tmpl_bin, cv2.TM_CCOEFF_NORMED)

# Suppress the legend swatch itself so the template doesn't trivially match
# against its own source pixels. The result map is (H-th+1) x (W-tw+1); the
# top-left coord that would perfectly overlap the swatch is (ty0, tx0).
mask_y0 = max(0, ty0 - th // 2)
mask_y1 = min(result.shape[0], ty0 + th // 2 + 1)
mask_x0 = max(0, tx0 - tw // 2)
mask_x1 = min(result.shape[1], tx0 + tw // 2 + 1)
result[mask_y0:mask_y1, mask_x0:mask_x1] = -1.0

hit_ys, hit_xs = np.where(result >= MATCH_THRESHOLD)
print(f"Raw hits at threshold {MATCH_THRESHOLD}: {len(hit_ys)}")

# Paint each hit's template footprint into a binary coverage mask. Adjacent
# hits naturally merge into wall-shaped blobs.
coverage = np.zeros(page_gray.shape, dtype=np.uint8)
for y, x in zip(hit_ys, hit_xs):
    coverage[y:y + th, x:x + tw] = 255

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (CLOSE_KERNEL, CLOSE_KERNEL))
coverage = cv2.morphologyEx(coverage, cv2.MORPH_CLOSE, kernel, iterations=CLOSE_ITERS)

raw_contours, _ = cv2.findContours(coverage, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
min_area = (th * tw) * MIN_POLY_AREA_FRAC

contours = []
for c in raw_contours:
    if cv2.contourArea(c) < min_area:
        continue
    contours.append(cv2.approxPolyDP(c, APPROX_EPS_PX, True))

print(f"Polygons after area filter + simplify: {len(contours)}")
if contours:
    areas = [cv2.contourArea(c) for c in contours]
    print(f"  Area range (px^2): {int(min(areas))} .. {int(max(areas))}")

In [ ]:
HIGHLIGHT_RGB = (0, 255, 0)
OUTLINE_RGB = (0, 180, 0)
HIGHLIGHT_ALPHA = 0.35
OUTLINE_THICKNESS = 2

annotated = page_rgb.copy()

if contours:
    fill_layer = page_rgb.copy()
    cv2.fillPoly(fill_layer, contours, color=HIGHLIGHT_RGB)
    annotated = cv2.addWeighted(
        annotated, 1.0 - HIGHLIGHT_ALPHA, fill_layer, HIGHLIGHT_ALPHA, 0
    )
    cv2.polylines(
        annotated,
        contours,
        isClosed=True,
        color=OUTLINE_RGB,
        thickness=OUTLINE_THICKNESS,
    )

png_path = out_dir / f"{drawing}_p{page_num}_{slug}.png"
json_path = out_dir / f"{drawing}_p{page_num}_{slug}.json"

Image.fromarray(annotated).save(png_path)

polygons_pdf = []
for c in contours:
    pts = c.reshape(-1, 2)
    polygons_pdf.append(
        [[float(x) / scale, float(y) / scale] for x, y in pts]
    )

payload = {
    "drawing": drawing,
    "page": page_num,
    "template_name": template_name,
    "template_rect_pdf": [
        rect_data["x0"],
        rect_data["top"],
        rect_data["x1"],
        rect_data["bottom"],
    ],
    "dpi": DPI,
    "scale": scale,
    "bin_threshold": BIN_THRESHOLD,
    "match_threshold": MATCH_THRESHOLD,
    "polygon_count": len(polygons_pdf),
    "polygons_pdf": polygons_pdf,
}
with open(json_path, "w") as f:
    json.dump(payload, f, indent=2)

print(f"Saved annotated PNG: {png_path}")
print(f"Saved polygons JSON: {json_path}  ({len(polygons_pdf)} polygons)")